In [ ]:
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.5.1+cu121.html

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 71.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 41.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 115.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 820.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196

In [ ]:
import numpy as np
from scipy.spatial import ConvexHull
import torch
from torch.nn import Sequential, Linear, ReLU
from torch_geometric.nn import global_max_pool
from torch_geometric.data import Data
from scipy.optimize import minimize

In [ ]:
def project_to_2d(vertices, theta, phi):
    R_z = np.array([
        [np.cos(theta), -np.sin(theta), 0],
        [np.sin(theta),  np.cos(theta), 0],
        [0,             0,              1]
    ])

    R_y = np.array([
        [ np.cos(phi), 0, np.sin(phi)],
        [ 0,           1, 0          ],
        [-np.sin(phi), 0, np.cos(phi)]
    ])

    rotated_3d = vertices @ (R_y @ R_z).T
    return rotated_3d[:, :2]

In [ ]:
def solve_containment_inequalities(vertices, faces, params):
    u, v, theta_p, phi_p, alpha, theta_q, phi_q = params

    Q_2d = project_to_2d(vertices, theta_q, phi_q)
    hull_Q = ConvexHull(Q_2d)

    A_mat = hull_Q.equations[:, :2]
    b_vec = -hull_Q.equations[:, 2]

    P_2d_raw = project_to_2d(vertices, theta_p, phi_p)

    R_alpha = np.array([
        [np.cos(alpha), -np.sin(alpha)],
        [np.sin(alpha),  np.cos(alpha)]
    ])

    P_2d = P_2d_raw @ R_alpha.T

    hull_P = ConvexHull(P_2d)
    P_verts = P_2d[hull_P.vertices]

    denom = A_mat @ P_verts.T

    num = b_vec[:, None] - A_mat @ np.array([u, v])[:, None]

    if np.any(num < 0):
        return 0.0

    valid_denoms = denom > 1e-8

    if not np.any(valid_denoms):
        return 0.0

    ratios = num / denom
    max_mu = np.min(ratios[valid_denoms])

    return max_mu

In [ ]:
def compute_nieuwland_constant(vertices, faces, num_restarts=25):
    """
    Module 2: 7D Optimization to find the exact Nieuwland constant.
    Now utilizes Multi-Start random initializations to escape the 1.0 local minimum.
    """
    best_mu = 0.0

    def objective_function(x):
        mu = solve_containment_inequalities(vertices, faces, x)
        return -mu

    # 1. Try the baseline trivial solution first
    baseline_result = minimize(objective_function, np.zeros(7), method='Nelder-Mead')
    best_mu = max(best_mu, -baseline_result.fun)

    # 2. Multi-start random initializations
    for _ in range(num_restarts):
        random_guess = np.random.uniform(low=-np.pi, high=np.pi, size=7)

        result = minimize(objective_function, random_guess, method='Nelder-Mead')

        found_mu = -result.fun
        if found_mu > best_mu:
            best_mu = found_mu

    return best_mu

In [ ]:
def generate_balanced_shape(N_samples=1024):
    strategy = np.random.choice(['high_density_spherical', 'perturbed_poles'])

    if strategy == 'high_density_spherical':
        points = np.random.randn(400, 3)
        points /= np.linalg.norm(points, axis=1)[:, np.newaxis]
        hull = ConvexHull(points)

    else:
        points = np.random.randn(500, 3)
        points /= np.linalg.norm(points, axis=1)[:, np.newaxis]

        z_mask = np.abs(points[:, 2]) > 0.85
        points[z_mask, 2] *= 0.92

        hull = ConvexHull(points)

    vertex_map = {original_idx: i for i, original_idx in enumerate(hull.vertices)}
    clean_vertices = hull.points[hull.vertices]

    clean_faces = []
    for face in hull.simplices:
        clean_faces.append([vertex_map[idx] for idx in face])
    clean_faces = np.array(clean_faces)

    vertices = clean_vertices
    faces_coords = vertices[clean_faces]

    areas = np.linalg.norm(np.cross(faces_coords[:,1] - faces_coords[:,0],
                                    faces_coords[:,2] - faces_coords[:,0]), axis=1) / 2.0
    area_probabilities = areas / np.sum(areas)

    chosen_face_indices = np.random.choice(len(faces_coords), size=N_samples, p=area_probabilities)
    sampled_points = np.zeros((N_samples, 3))

    for i, face_idx in enumerate(chosen_face_indices):
        r1, r2 = np.random.rand(2)
        if r1 + r2 > 1:
            r1, r2 = 1 - r1, 1 - r2
        a, b, c = faces_coords[face_idx]
        sampled_points[i] = (1 - r1 - r2) * a + r1 * b + r2 * c

    centroid = np.mean(sampled_points, axis=0)
    centered = sampled_points - centroid
    scale = np.max(np.linalg.norm(centered, axis=1))
    normalized_points = centered / scale

    return normalized_points, vertices, clean_faces

In [ ]:
def compute_sphericity(vertices):
    from scipy.spatial import ConvexHull
    import numpy as np

    hull = ConvexHull(vertices)

    if hull.area == 0:
        return 0.0

    sphericity = (np.pi ** (1.0 / 3.0) * (6.0 * hull.volume) ** (2.0 / 3.0)) / hull.area
    return sphericity

def build_and_save_dataset(num_samples=3000, filename='rupert_dataset.pt'):
    data_list = []
    print(f"Generating {num_samples} balanced shapes. This will take time...")

    for i in range(num_samples):
        point_cloud, vertices, faces = generate_balanced_shape()

        mu_label = compute_nieuwland_constant(vertices, faces, num_restarts=10)
        psi_label = compute_sphericity(vertices)

        x_tensor = torch.tensor(point_cloud, dtype=torch.float)
        y_tensor = torch.tensor([[mu_label, psi_label]], dtype=torch.float)

        data_list.append(Data(x=x_tensor, pos=x_tensor, y=y_tensor))

        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{num_samples}...")

    torch.save(data_list, filename)
    print(f"Balanced dataset saved to {filename}!")

In [ ]:
if __name__ == "__main__":
    build_and_save_dataset()

Generating 3000 balanced shapes. This will take time...
Processed 50/3000...
Processed 100/3000...
Processed 150/3000...
Processed 200/3000...
Processed 250/3000...
Processed 300/3000...
Processed 350/3000...
Processed 400/3000...
Processed 450/3000...
Processed 500/3000...
Processed 550/3000...
Processed 600/3000...
Processed 650/3000...
Processed 700/3000...
Processed 750/3000...
Processed 800/3000...
Processed 850/3000...
Processed 900/3000...
Processed 950/3000...
Processed 1000/3000...
Processed 1050/3000...
Processed 1100/3000...
Processed 1150/3000...
Processed 1200/3000...
Processed 1250/3000...
Processed 1300/3000...
Processed 1350/3000...
Processed 1400/3000...
Processed 1450/3000...
Processed 1500/3000...
Processed 1550/3000...
Processed 1600/3000...
Processed 1650/3000...
Processed 1700/3000...
Processed 1750/3000...
Processed 1800/3000...
Processed 1850/3000...
Processed 1900/3000...
Processed 1950/3000...
Processed 2000/3000...
Processed 2050/3000...
Processed 2100/3000..